In [1]:
import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely import LineString, Point
from shapely.ops import substring, unary_union

from iduedu import config

logger = config.logger

def keep_largest_strongly_connected_component(graph: nx.MultiDiGraph, relabel_nodes: bool = True) -> nx.MultiDiGraph:
    graph = nx.MultiDiGraph(graph)  # Копируем как MultiDiGraph
    weakly_connected_components = list(nx.weakly_connected_components(graph))
    if len(weakly_connected_components) > 1:
        logger.warning(
            f"Graph contains {len(weakly_connected_components)} weakly connected components. "
            f"Component sizes: {[len(c) for c in weakly_connected_components]}"
        )
    all_scc = sorted(nx.strongly_connected_components(graph), key=len)
    nodes_to_del = set().union(*all_scc[:-1])
    if nodes_to_del:
        logger.warning(
            f"Removing {len(nodes_to_del)} nodes from {len(all_scc) - 1} smaller components. "
            f"Retaining largest component ({len(all_scc[-1])} nodes)."
        )
        graph.remove_nodes_from(nodes_to_del)
        if relabel_nodes:
            graph = nx.convert_node_labels_to_integers(graph)
    return graph

def join_graph(
    public_transport_g: nx.DiGraph,
    walk_g: nx.MultiDiGraph,
    max_dist: float = 20,
    retain_all: bool = False,
    tolerance: float = 3
) -> nx.MultiDiGraph:
    """
    Объединяет графы общественного транспорта и пешеходной сети.
    Удаляет узлы типа 'platform' и ребра типа 'boarding', перенаправляя связи.
    Объединяет остановки с близкими координатами, сохраняя маршруты.
    Проецирует остановки на пешеходные ребра.
    
    Args:
        public_transport_g: Граф общественного транспорта (DiGraph).
        walk_g: Граф пешеходной сети (MultiDiGraph).
        max_dist: Максимальное расстояние для проекции (метры).
        retain_all: Сохранять все компоненты связности или только крупнейшую.
        tolerance: Радиус буфера для объединения близких остановок (метры).
    
    Returns:
        Межмодальный граф (MultiDiGraph).
    """
    def merge_routes(route_series):
        all_routes = []
        for r in route_series:
            if isinstance(r, list):
                all_routes.extend(r)
            elif r is not None:
                all_routes.append(r)
        return list(dict.fromkeys(all_routes))  # Уникальные маршруты с сохранением порядка

    assert public_transport_g.graph["crs"] == walk_g.graph["crs"], "CRS mismatching."
    logger.info("Composing intermodal graph...")

    # Релейблинг узлов для уникальности
    num_nodes_g1 = len(public_transport_g.nodes)
    mapping_g1 = {node: idx for idx, node in enumerate(public_transport_g.nodes)}
    mapping_g2 = {node: idx + num_nodes_g1 for idx, node in enumerate(walk_g.nodes)}
    transport = nx.relabel_nodes(nx.MultiDiGraph(public_transport_g), mapping_g1)
    walk = nx.relabel_nodes(walk_g, mapping_g2)

    # Шаг 1: Перенаправляем boarding-рёбра напрямую между остановками
    logger.debug("Redirecting boarding edges through platforms")
    platforms_df = pd.DataFrame.from_dict(dict(transport.nodes(data=True)), orient="index")
    platform_nodes = platforms_df[platforms_df["type"] == "platform"].index.tolist()
    
    # Собираем группы остановок, связанных через одну платформу
    stop_groups = []
    for platform in platform_nodes:
        connected_stops = set()
        for pred in transport.predecessors(platform):
            if transport.nodes[pred].get("type") != "platform":
                connected_stops.add(pred)
        for succ in transport.successors(platform):
            if transport.nodes[succ].get("type") != "platform":
                connected_stops.add(succ)
        
        if len(connected_stops) > 1:
            stop_groups.append(list(connected_stops))
    
    # Шаг 2: Соединяем остановки из каждой группы напрямую
    for stops in stop_groups:
        main_stop = stops[0]
        logger.debug(f"Merging {len(stops)} stops connected by platform: {stops}")
        
        for dup_stop in stops[1:]:
            if dup_stop not in transport:
                logger.debug(f"Skipping {dup_stop} - not in graph")
                continue
                
            # Перенаправляем входящие рёбра
            for pred in list(transport.predecessors(dup_stop)):
                edge_data_dict = transport.get_edge_data(pred, dup_stop)
                if edge_data_dict:
                    logger.debug(f"Incoming edges {pred} -> {dup_stop}: {len(edge_data_dict)} edges")
                    for key, edge_data in edge_data_dict.items():
                        logger.debug(f"Copying edge {pred} -> {dup_stop} (key={key}): {edge_data}")
                        transport.add_edge(pred, main_stop, **edge_data)
            
            # Перенаправляем исходящие рёбра
            for succ in list(transport.successors(dup_stop)):
                edge_data_dict = transport.get_edge_data(dup_stop, succ)
                if edge_data_dict:
                    logger.debug(f"Outgoing edges {dup_stop} -> {succ}: {len(edge_data_dict)} edges")
                    for key, edge_data in edge_data_dict.items():
                        logger.debug(f"Copying edge {dup_stop} -> {succ} (key={key}): {edge_data}")
                        transport.add_edge(main_stop, succ, **edge_data)
            
            # Объединяем маршруты
            main_routes = transport.nodes[main_stop].get("route", [])
            dup_routes = transport.nodes[dup_stop].get("route", [])
            if isinstance(main_routes, list) and isinstance(dup_routes, list):
                combined = main_routes + dup_routes
                transport.nodes[main_stop]["route"] = list(dict.fromkeys(combined))
            
            # Удаляем дубликат
            logger.debug(f"Removing duplicate stop: {dup_stop}")
            transport.remove_node(dup_stop)
    
    # Шаг 3: Удаляем платформы и все boarding-рёбра
    transport.remove_nodes_from(platform_nodes)
    boarding_edges = [(u, v, key) for u, v, key, data in transport.edges(keys=True, data=True) 
                      if data.get("type") == "boarding"]
    transport.remove_edges_from(boarding_edges)
    
    logger.debug(f"Removed {len(platform_nodes)} platform nodes and {len(boarding_edges)} boarding edges. "
                 f"Merged {len(stop_groups)} groups of stops")

    # Обновляем platforms после удаления дубликатов
    platforms = pd.DataFrame.from_dict(dict(transport.nodes(data=True)), orient="index")
    platforms['node_id'] = platforms.index
    platforms["geometry"] = gpd.points_from_xy(platforms["x"], platforms["y"])
    platforms = gpd.GeoDataFrame(platforms, crs=transport.graph["crs"])

    # Создаём GeoDataFrame рёбер пешеходного графа
    logger.debug("Creating walk edges GeoDataFrame")
    walk_edges = gpd.GeoDataFrame(
        nx.to_pandas_edgelist(walk, source="u", target="v", edge_key="k"), 
        crs=walk.graph["crs"]
    )
    walk_edges["edge_geometry"] = walk_edges["geometry"]

    # Очистка от служебных колонок
    platforms = platforms.reset_index(drop=True)
    for col in ['index_right', 'index']:
        if col in platforms.columns:
            platforms = platforms.drop(columns=[col])
    walk_edges = walk_edges.reset_index(drop=True)
    for col in ['index_right', 'index']:
        if col in walk_edges.columns:
            walk_edges = walk_edges.drop(columns=[col])

    # Spatial join
    logger.debug("Performing spatial join")
    projection_join = platforms.sjoin_nearest(walk_edges, max_distance=max_dist, distance_col="dist")

    if projection_join.empty:
        logger.warning("No stops found within max_distance of walk edges!")
        return nx.compose(nx.MultiDiGraph(transport), nx.MultiDiGraph(walk))

    # Проекция на рёбра
    projection_join["project_dist"] = projection_join["edge_geometry"].project(projection_join["geometry"])
    projection_join["project_point"] = gpd.GeoSeries(
        projection_join["edge_geometry"].interpolate(projection_join["project_dist"]), 
        crs=walk.graph["crs"]
    ).set_precision(1)

    # Группируем по точкам проекции
    logger.debug("Grouping by projection points")
    projection_join = projection_join.groupby(by="project_point", as_index=False).agg({
        "node_id": lambda x: list(x),
        "index_right": "first",
        "geometry": lambda x: list(x),
        "u": "first",
        "v": "first",
        "k": "first",
        "project_dist": "first",
        "route": merge_routes,
    })

    # Объединяем остановки с одинаковыми точками проекции
    for ind, row in projection_join.iterrows():
        if not isinstance(row["node_id"], list) or len(row["node_id"]) == 0:
            continue
        main_node = row["node_id"][0]
        if len(row["node_id"]) > 1:
            logger.debug(f"Collapsing {len(row['node_id'])} stops at projection point: {row['node_id']}")
        for dup_node in row["node_id"][1:]:
            if dup_node == main_node or dup_node not in transport:
                continue
            for pred in list(transport.predecessors(dup_node)):
                edge_data_dict = transport.get_edge_data(pred, dup_node)
                if edge_data_dict:
                    for key, edge_data in edge_data_dict.items():
                        logger.debug(f"Copying edge {pred} -> {dup_node} (key={key}): {edge_data}")
                        transport.add_edge(pred, main_node, **edge_data)
            for succ in list(transport.successors(dup_node)):
                edge_data_dict = transport.get_edge_data(dup_node, succ)
                if edge_data_dict:
                    for key, edge_data in edge_data_dict.items():
                        logger.debug(f"Copying edge {dup_node} -> {succ} (key={key}): {edge_data}")
                        transport.add_edge(main_node, succ, **edge_data)
            transport.remove_node(dup_node)
        transport.nodes[main_node]["route"] = row["route"]
        projection_join.at[ind, "node_id"] = main_node

    # Группируем по рёбрам для проекции
    logger.debug("Grouping by edges")
    points_grouped_by_edge = projection_join.groupby(by="index_right", as_index=False).agg({
        "node_id": lambda x: list(x) if isinstance(x, (list, pd.Series)) else [x],
        "geometry": lambda x: list(x.iloc[0]) if isinstance(x.iloc[0], list) else [x.iloc[0]],
        "u": "first",
        "v": "first",
        "k": "first",
        "route": merge_routes,
    })

    # Получаем скорость ходьбы
    try:
        speed = walk.graph["walk_speed"]
    except KeyError:
        logger.warning("No walk_speed in graph, using default: 83.33 m/min")
        speed = 83.33

    edges_to_del = []

    # Проецируем остановки на рёбра
    logger.debug("Projecting stops onto walk edges")
    for i in range(len(points_grouped_by_edge)):
        row = points_grouped_by_edge.iloc[i]
        u, v, k = row["u"], row["v"], row["k"]
        edge_data = walk.get_edge_data(u, v, k) or walk.get_edge_data(v, u, k)
        if not edge_data:
            logger.error(f"Edge geometry not found for (u={u}, v={v}, k={k})")
            continue
        edge = edge_data.get("geometry") or edge_data.get("edge_geometry")
        if edge is None:
            logger.error(f"Edge geometry not found for (u={u}, v={v}, k={k})")
            continue

        if len(row["node_id"]) == 1:
            platform_id = row["node_id"][0]
            geom = row["geometry"][0] if isinstance(row["geometry"], list) else row["geometry"]
            dist = edge.project(geom)
            projected_point = edge.interpolate(dist)
            if dist == 0:
                nx.relabel_nodes(walk, {u: platform_id}, copy=False)
                points_grouped_by_edge.loc[points_grouped_by_edge["u"] == u, "u"] = platform_id
                points_grouped_by_edge.loc[points_grouped_by_edge["v"] == u, "v"] = platform_id
            elif dist == edge.length:
                nx.relabel_nodes(walk, {v: platform_id}, copy=False)
                points_grouped_by_edge.loc[points_grouped_by_edge["u"] == v, "u"] = platform_id
                points_grouped_by_edge.loc[points_grouped_by_edge["v"] == v, "v"] = platform_id
            else:
                line1 = substring(edge, 0, dist)
                line2 = substring(edge, dist, edge.length)
                edges_to_del.append((u, v, k))
                if walk.has_edge(v, u, k):
                    edges_to_del.append((v, u, k))
                walk.add_node(platform_id, x=round(projected_point.x, 5), y=round(projected_point.y, 5))
                walk.add_edge(u, platform_id, geometry=line1, length_meter=round(line1.length, 3),
                             time_min=round(line1.length / speed, 3), type="walk")
                walk.add_edge(platform_id, u, geometry=line1.reverse(), length_meter=round(line1.length, 3),
                             time_min=round(line1.length / speed, 3), type="walk")
                walk.add_edge(platform_id, v, geometry=line2, length_meter=round(line2.length, 3),
                             time_min=round(line2.length / speed, 3), type="walk")
                walk.add_edge(v, platform_id, geometry=line2.reverse(), length_meter=round(line2.length, 3),
                             time_min=round(line2.length / speed, 3), type="walk")
        else:
            dist_project = []
            for idx, geom in zip(row["node_id"], row["geometry"]):
                if isinstance(geom, (list, tuple)):
                    geom = geom[0] if geom else None
                if geom:
                    dist = edge.project(geom)
                    dist_project.append((dist, edge.interpolate(dist), idx))
            dist_project.sort(key=lambda x: x[0])
            u_orig, v_orig = u, v
            last_dist = 0
            last_u = u
            for dist, projected_point, cur_index in dist_project:
                if dist == 0:
                    nx.relabel_nodes(walk, {u: cur_index}, copy=False)
                    u_orig = cur_index
                    points_grouped_by_edge.loc[points_grouped_by_edge["u"] == u, "u"] = cur_index
                    points_grouped_by_edge.loc[points_grouped_by_edge["v"] == u, "v"] = cur_index
                elif dist == edge.length:
                    nx.relabel_nodes(walk, {v: cur_index}, copy=False)
                    v_orig = cur_index
                    points_grouped_by_edge.loc[points_grouped_by_edge["u"] == v, "u"] = cur_index
                    points_grouped_by_edge.loc[points_grouped_by_edge["v"] == v, "v"] = cur_index
                else:
                    line = substring(edge, last_dist, dist)
                    if isinstance(line, Point):
                        logger.warning(f"Line segment is a Point at dist {dist}, skipping")
                        continue
                    walk.add_node(cur_index, x=round(projected_point.x, 5), y=round(projected_point.y, 5))
                    walk.add_edge(last_u, cur_index, geometry=line, length_meter=round(line.length, 3),
                                 time_min=round(line.length / speed, 3), type="walk")
                    walk.add_edge(cur_index, last_u, geometry=line.reverse(), length_meter=round(line.length, 3),
                                 time_min=round(line.length / speed, 3), type="walk")
                    last_u = cur_index
                last_dist = dist
            if last_dist < edge.length:
                line = substring(edge, last_dist, edge.length)
                walk.add_edge(last_u, v, geometry=line, length_meter=round(line.length, 3),
                             time_min=round(line.length / speed, 3), type="walk")
                walk.add_edge(v, last_u, geometry=line.reverse(), length_meter=round(line.length, 3),
                             time_min=round(line.length / speed, 3), type="walk")
            edges_to_del.append((u_orig, v_orig, k))
            if walk.has_edge(v_orig, u_orig, k):
                edges_to_del.append((v_orig, u_orig, k))

    # Удаляем старые рёбра
    walk.remove_edges_from(edges_to_del)

    # Объединяем графы
    logger.debug("Composing graphs")
    intermodal = nx.compose(nx.MultiDiGraph(transport), nx.MultiDiGraph(walk))
    
    if not retain_all:
        intermodal = keep_largest_strongly_connected_component(intermodal)
    
    # Удаляем узлы без координат (закомментировано, как в оригинале)
    # intermodal.remove_nodes_from([node for node, data in intermodal.nodes(data=True) if "x" not in data])
    
    # Финальный релейблинг
    logger.debug("Final relabeling")
    mapping = {old_label: new_label for new_label, old_label in enumerate(intermodal.nodes())}
    nx.relabel_nodes(intermodal, mapping, copy=False)
    intermodal.graph["type"] = "intermodal"
    
    logger.info(f"Intermodal graph created: {intermodal.number_of_nodes()} nodes, {intermodal.number_of_edges()} edges")
    return intermodal

In [6]:
from iduedu import get_drive_graph
from iduedu import get_all_public_transport_graph
# Get territory boundary
from iduedu import get_boundary
from iduedu import graph_to_gdf

vologda = get_boundary(1327509)
G_drive = get_drive_graph(polygon=vologda) 
G_pt = get_all_public_transport_graph(polygon=vologda)

2025-10-07 15:42:20.884 | INFO     | iduedu.modules.drive_walk_builder:get_drive_graph_by_poly:64 - Downloading drive graph from OSM, it may take a while for large territory ...
2025-10-07 15:42:23.696 | WARNING  | iduedu.utils.utils:keep_largest_strongly_connected_component:37 - Removing 4 nodes from 3 smaller strongly connected components. These are subgraphs where nodes are internally reachable but isolated from the rest. Retaining only the largest strongly connected component (1354 nodes).


RequestError: Request failed with status code 429, reason: Too Many Requests (status: 429, reason: Too Many Requests).

In [ ]:
import logging

# logging.basicConfig(level=logging.ERROR)
# logger = logging.getLogger(__name__)

G = join_graph(G_pt, G_drive, retain_all=False)

2025-10-06 12:13:11.392 | INFO     | __main__:join_graph:64 - Composing intermodal graph...
2025-10-06 12:13:11.566 | WARNING  | __main__:join_graph:233 - No walk_speed in graph, using default: 83.33 m/min
2025-10-06 12:13:11.592 | WARNING  | __main__:join_graph:306 - Line segment is a Point at dist 297.98793474235913, skipping
2025-10-06 12:13:11.602 | WARNING  | __main__:join_graph:306 - Line segment is a Point at dist 36.04824588138875, skipping
2025-10-06 12:13:11.687 | WARNING  | __main__:keep_largest_strongly_connected_component:22 - Removing 4 nodes from 4 smaller components. Retaining largest component (1688 nodes).
2025-10-06 12:13:11.723 | INFO     | __main__:join_graph:344 - Intermodal graph created: 1688 nodes, 5050 edges


In [ ]:
import networkx as nx
from collections import defaultdict


# --- 1️⃣ Собираем рёбра, у которых есть маршрут ---
routes_by_name = defaultdict(list)
for u, v, key, data in G.edges(keys=True, data=True):
    route_name = data.get("route")
    if route_name:  # пропускаем рёбра без маршрута
        routes_by_name[route_name].append((u, v, key))

results = []

# --- 2️⃣ Для каждого маршрута восстанавливаем последовательность узлов ---
for route_name, route_edges in routes_by_name.items():
    # создаём подграф маршрута
    subG = nx.Graph()
    for u, v, key in route_edges:
        subG.add_edge(u, v, key=key)

    # определяем начала/концы маршрута
    degrees = dict(subG.degree())
    endpoints = [n for n, d in degrees.items() if d == 1]

    if endpoints:
        start = endpoints[0]
    else:
        # кольцевой маршрут
        start = list(subG.nodes())[0]

    # проходим маршрут в порядке связей (DFS)
    path_nodes = list(nx.dfs_preorder_nodes(subG, source=start))

    results.append({
        "name": route_name,
        "nodes": path_nodes
    })

# --- 3️⃣ Вывод ---
for r in results:
    print(f"Маршрут: {r['name']}")
    print("Последовательность узлов:", r["nodes"], "\n")


Маршрут: Автобус №37Э: Молочное — Вологда
Последовательность узлов: [0, 3, 1, 4, 6, 9, 11, 36, 41, 68, 100, 120, 114, 109] 

Маршрут: Автобус №37: Молочное - Вологда
Последовательность узлов: [1, 4, 6, 9, 11, 29, 35, 45, 64, 95, 113, 115, 124, 141, 145, 152, 148, 156] 

Маршрут: Автобус №37Э: Вологда — Молочное
Последовательность узлов: [0, 2, 5, 7, 8, 12, 38, 44, 73, 91, 101, 119, 112, 109] 

Маршрут: Автобус №37: Вологда - Молочное
Последовательность узлов: [1, 5, 7, 8, 12, 30, 34, 43, 55, 90, 98, 116, 122, 140, 171, 156] 

Маршрут: Автобус №42: Екимцево => Дальняя улица
Последовательность узлов: [10, 13, 16, 17, 19, 22, 23, 25, 27, 31, 36, 40, 41, 49, 54, 68, 79, 89, 97, 103, 127, 141, 159, 181, 222, 236, 250, 256, 260, 284, 300, 305, 301, 296] 

Маршрут: Автобус №42: Дальняя улица => Екимцево
Последовательность узлов: [10, 14, 15, 18, 20, 21, 24, 26, 28, 33, 38, 39, 52, 73, 80, 91, 102, 104, 123, 142, 161, 184, 224, 243, 251, 261, 273, 299, 304, 303, 297] 

Маршрут: Автобус №16: ВП

In [ ]:
# print("Исходный граф:")
# print(f"Узлов: {G.number_of_nodes()}, Ребер: {G.number_of_edges()}\n")

# # Очищаем граф
# G = clean_graph(G)

# print(f"\nОчищенный граф:")
# print(f"Узлов: {G.number_of_nodes()}, Ребер: {G.number_of_edges()}")

Исходный граф:
Узлов: 2014, Ребер: 4863

Удалено 339 узлов типа 'platform'
Удалено 0 ребер типа 'boarding'

Очищенный граф:
Узлов: 1675, Ребер: 4185


In [55]:
gdf = graph_to_gdf(G, restore_edge_geom=True)
gdf['route'].fillna(0,inplace=True)
gdf['index'] = gdf.index

# посмотреть отдельно маршруты
gdf_routes = gdf[gdf['route']!=0]

# посмотреть отдельно дороги
gdf_roads = gdf[gdf['route']==0]

gdf = gdf.to_crs(4326)

/tmp/ipykernel_1278/1856104621.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  gdf['route'].fillna(0,inplace=True)


In [62]:
import folium
from folium import plugins
import pandas as pd
import geopandas as gpd
from collections import defaultdict
import networkx as nx

# ============================================
# 1. ВОССТАНОВЛЕНИЕ ПОСЛЕДОВАТЕЛЬНОСТИ МАРШРУТОВ
# ============================================

def extract_routes(G):
    """Извлекает маршруты из графа и возвращает их последовательности"""
    routes_by_name = defaultdict(list)
    
    # Собираем рёбра по маршрутам
    for u, v, key, data in G.edges(keys=True, data=True):
        route_name = data.get("route")
        if route_name:
            routes_by_name[route_name].append((u, v, key))
    
    results = []
    
    # Восстанавливаем последовательность для каждого маршрута
    for route_name, route_edges in routes_by_name.items():
        subG = nx.Graph()
        for u, v, key in route_edges:
            subG.add_edge(u, v, key=key)
        
        # Находим конечные точки
        degrees = dict(subG.degree())
        endpoints = [n for n, d in degrees.items() if d == 1]
        
        if endpoints:
            start = endpoints[0]
        else:
            start = list(subG.nodes())[0]
        
        # Получаем последовательность узлов
        path_nodes = list(nx.dfs_preorder_nodes(subG, source=start))
        
        results.append({
            "name": route_name,
            "nodes": path_nodes
        })
    
    return results

# ============================================
# 2. СОЗДАНИЕ ИНТЕРАКТИВНОЙ КАРТЫ
# ============================================

def create_route_map(routes, nodes_gdf, edges_gdf, output_file='routes_map.html'):
    """
    Создает интерактивную карту маршрутов с возможностью выбора
    
    Параметры:
    - routes: список словарей с маршрутами (результат extract_routes)
    - nodes_gdf: GeoDataFrame с точками (узлами)
    - edges_gdf: GeoDataFrame с ребрами
    - output_file: имя выходного HTML файла
    """
    
    # Определяем центр карты
    center_lat = nodes_gdf.geometry.y.mean()
    center_lon = nodes_gdf.geometry.x.mean()
    
    # Создаем базовую карту
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='OpenStreetMap'
    )
    
    # Добавляем дополнительные слои карты
    folium.TileLayer('CartoDB positron', name='Light Map').add_to(m)
    folium.TileLayer('CartoDB dark_matter', name='Dark Map').add_to(m)
    
    # Цветовая палитра для маршрутов
    colors = [
        '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
        '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe',
        '#008080', '#e6beff', '#9a6324', '#fffac8', '#800000',
        '#aaffc3', '#808000', '#ffd8b1', '#000075', '#808080'
    ]
    
    # Создаем группы для каждого маршрута
    route_groups = {}
    
    for idx, route in enumerate(routes):
        route_name = route['name']
        route_nodes = route['nodes']
        color = colors[idx % len(colors)]
        
        # Создаем feature group для маршрута (изначально выключен)
        fg = folium.FeatureGroup(name=route_name, show=False)
        
        # Фильтруем рёбра, относящиеся к данному маршруту
        route_edges = edges_gdf[edges_gdf['route'] == route_name].copy()
        
        if len(route_edges) > 0:
            # Добавляем линии маршрута
            for _, edge in route_edges.iterrows():
                if edge.geometry is not None:
                    # Извлекаем координаты
                    coords = list(edge.geometry.coords)
                    # Меняем местами x и y для Folium (lat, lon)
                    folium_coords = [(lat, lon) for lon, lat in coords]
                    
                    # Создаем popup с информацией о ребре
                    popup_text = f"<b>Маршрут:</b> {route_name}<br>"
                    if 'street_count' in edge and pd.notna(edge['street_count']):
                        popup_text += f"<b>Улица:</b> {edge['street_count']}<br>"
                    if 'length_meter' in edge and pd.notna(edge['length_meter']):
                        popup_text += f"<b>Длина:</b> {edge['length_meter']:.1f} м<br>"
                    
                    folium.PolyLine(
                        folium_coords,
                        color=color,
                        weight=4,
                        opacity=0.8,
                        popup=folium.Popup(popup_text, max_width=300)
                    ).add_to(fg)
            
            # Добавляем узлы маршрута
            route_nodes_gdf = nodes_gdf[nodes_gdf.index.isin(route_nodes)]
            
            for idx_node, node in route_nodes_gdf.iterrows():
                if node.geometry is not None:
                    # Определяем, является ли узел конечной точкой
                    is_endpoint = (idx_node == route_nodes[0] or 
                                 idx_node == route_nodes[-1])
                    
                    popup_text = f"<b>Маршрут:</b> {route_name}<br>"
                    popup_text += f"<b>Узел:</b> {idx_node}<br>"
                    
                    if is_endpoint:
                        # Конечные точки - большие маркеры
                        folium.CircleMarker(
                            location=[node.geometry.y, node.geometry.x],
                            radius=8,
                            color=color,
                            fill=True,
                            fillColor=color,
                            fillOpacity=0.9,
                            popup=folium.Popup(popup_text, max_width=200)
                        ).add_to(fg)
                    else:
                        # Промежуточные точки - маленькие маркеры
                        folium.CircleMarker(
                            location=[node.geometry.y, node.geometry.x],
                            radius=3,
                            color=color,
                            fill=True,
                            fillColor='white',
                            fillOpacity=0.7,
                            popup=folium.Popup(popup_text, max_width=200)
                        ).add_to(fg)
        
        fg.add_to(m)
        route_groups[route_name] = fg
    
    # Добавляем контроль слоев (для выбора маршрутов)
    folium.LayerControl(collapsed=False).add_to(m)
    
    # Добавляем плагины для удобства
    plugins.Fullscreen().add_to(m)
    plugins.MeasureControl(position='topleft').add_to(m)
    plugins.LocateControl().add_to(m)
    
    # Сохраняем карту
    m.save(output_file)
    print(f"Карта сохранена в файл: {output_file}")
    
    return m


# ============================================
# 3. ИСПОЛЬЗОВАНИЕ
# ============================================

# Пример использования:
# 
# # Извлекаем маршруты из графа
routes = extract_routes(G)
nodes_gdf = gdf[gdf.geometry.type == 'Point']
edges_gdf = gdf[gdf.geometry.type == 'LineString']

# Создаем визуализацию
create_route_map(
    routes=routes,
    nodes_gdf=nodes_gdf,  # ваш GeoDataFrame с точками
    edges_gdf=edges_gdf,  # ваш GeoDataFrame с ребрами
    output_file='bus_routes_map.html'
)

# Вывод информации о маршрутах
for r in routes:
    print(f"Маршрут: {r['name']}")
    print(f"Количество остановок: {len(r['nodes'])}")
    print(f"Последовательность узлов: {r['nodes']}\n")

Карта сохранена в файл: bus_routes_map.html
Маршрут: Автобус №37Э: Молочное — Вологда
Количество остановок: 14
Последовательность узлов: [0, 3, 1, 4, 6, 9, 11, 36, 41, 68, 100, 120, 114, 109]

Маршрут: Автобус №37: Молочное - Вологда
Количество остановок: 18
Последовательность узлов: [1, 4, 6, 9, 11, 29, 35, 45, 64, 95, 113, 115, 124, 141, 145, 152, 148, 156]

Маршрут: Автобус №37Э: Вологда — Молочное
Количество остановок: 14
Последовательность узлов: [0, 2, 5, 7, 8, 12, 38, 44, 73, 91, 101, 119, 112, 109]

Маршрут: Автобус №37: Вологда - Молочное
Количество остановок: 16
Последовательность узлов: [1, 5, 7, 8, 12, 30, 34, 43, 55, 90, 98, 116, 122, 140, 171, 156]

Маршрут: Автобус №42: Екимцево => Дальняя улица
Количество остановок: 34
Последовательность узлов: [10, 13, 16, 17, 19, 22, 23, 25, 27, 31, 36, 40, 41, 49, 54, 68, 79, 89, 97, 103, 127, 141, 159, 181, 222, 236, 250, 256, 260, 284, 300, 305, 301, 296]

Маршрут: Автобус №42: Дальняя улица => Екимцево
Количество остановок: 31
Пос

In [58]:
target_routes = [
    "Автобус №37: Вологда - Молочное",
    "Автобус №37: Молочное - Вологда",
    # "Автобус №37Э: Молочное — Вологда"
]

mask = gdf["route"].apply(
    lambda x: (
        (x == 1) or (
            any(r in x for r in target_routes) if isinstance(x, (list, tuple)) 
            else x in target_routes
        )
    )
)

subset = gdf[mask]
subset.explore('route')

In [23]:
from iduedu import get_adj_matrix_gdf_to_gdf

points = gdf[gdf.geometry.type == 'Point']

matrix = get_adj_matrix_gdf_to_gdf(points, points, G, weight='length_meter')

In [26]:
matrix.loc[0,1]

np.float16(1092.0)

In [27]:
(matrix*60).loc[0,1]

np.float16(inf)

In [16]:
speed_kmh = 20
matrix * 60.0 / (1000.0 * float(speed_kmh))

/root/TNDP_learning/venv/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/root/TNDP_learning/venv/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,0,1,2,3,4,5,6,7,8,9,...,1678,1679,1680,1681,1682,1683,1684,1685,1686,1687
0,0.0,inf,inf,1.842773,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
1,inf,0.000000,inf,inf,2.765625,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
2,inf,inf,0.0,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
3,inf,1.432617,inf,0.000000,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
4,inf,inf,inf,inf,0.000000,inf,2.216797,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,0.000000,0.389893,inf,inf,inf
1684,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,0.389893,0.000000,inf,inf,inf
1685,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,2.968750,2.802734,inf,inf,inf,inf,inf,0.000000,0.043793,inf
1686,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,3.261719,3.095703,inf,inf,inf,inf,inf,0.292236,0.000000,inf


In [17]:
def count_connected_components(weight_matrix):
    n = len(weight_matrix)
    visited = [False] * n

    def dfs(v):
        visited[v] = True
        for u in range(n):
            # считаем, что есть ребро, если вес > 0 (или не бесконечность)
            if weight_matrix[v][u] != 0 and not visited[u]:
                dfs(u)

    components = 0
    for i in range(n):
        if not visited[i]:
            components += 1
            dfs(i)

    return components

components = count_connected_components(matrix * 60.0 / (1000.0 * float(speed_kmh)))



In [7]:
import pickle

with open("G_routes_Kemerovo_connectivity.pkl", "rb") as f:
    G_Kem = pickle.load(f)

In [ ]:
print(list(G.edges(data=True))[0])
print(list(G_Kem.edges(data=True))[0])

(0, 336, {'route': 'Автобус №37Э: Молочное — Вологда', 'type': 'boarding', 'geometry': nan, 'length_meter': 0.0, 'time_min': 0.0})
(0, 34, {'route_id': 3, 'segment_ix': 5, 'uv_from': (34, 0), 'weight': 608.3955300361124, 'time_min': 1.8251865901083373, 'geometry': <LINESTRING (445965.264 6134042.623, 446507.382 6134318.763)>, 'original_path': [(445965.26418797113, 6134042.623373404), (446507.3824675598, 6134318.7626351975)]})


In [9]:
print(list(G.nodes(data=True))[0])
print(list(G_Kem.nodes(data=True))[0])

(0, {'x': 546724.76411, 'y': 6561241.06335, 'type': 'platform', 'route': ['Автобус №37Э: Молочное — Вологда']})
(0, {'x': 445965.26418797113, 'y': 6134042.623373404, 'nodeID': 0, 'is_stop': True})


In [33]:
import folium
import geopandas as gpd
from shapely.geometry import Point
from shapely import affinity

intermodal = G

# --- подготовка GeoDataFrame ---
edges = gpd.GeoDataFrame(
    [
        {
            "u": u,
            "v": v,
            "geometry": d["geometry"],
            "route": d.get("route", "default")
        }
        for u, v, d in intermodal.edges(data=True) if "geometry" in d
    ],
    crs=intermodal.graph.get("crs")
).to_crs(epsg=4326)
# edges = edges[edges['route'] != 'default']
# --- список цветов (зациклим при необходимости) ---
color_palette = [
    "red", "blue", "green", "orange", "purple", "brown", "pink", "gray", "cyan", "magenta"
]

unique_routes = list(edges["route"].unique())
route_colors = {route: color_palette[i % len(color_palette)] for i, route in enumerate(unique_routes)}

# --- функция сдвига геометрии ---
def shifted_geometry(row, step=0.000):
    r = str(row["route"])
    shift_index = unique_routes.index(r) % 5 - 2  # значения от -2 до 2
    return affinity.translate(row["geometry"], xoff=0, yoff=shift_index * step)

edges["geometry"] = edges.apply(shifted_geometry, axis=1)

# --- узлы с координатами ---
nodes = gpd.GeoDataFrame(
    [
        {
            "node": n,
            "geometry": Point(d["x"], d["y"]),
            "type": d.get("type", None),
            "route": d.get("route", None)
        }
        for n, d in intermodal.nodes(data=True) if "x" in d and "y" in d
    ],
    crs=intermodal.graph.get("crs")
).to_crs(epsg=4326)


# --- карта ---
center = [nodes.geometry.y.mean(), nodes.geometry.x.mean()]
m = folium.Map(location=center, zoom_start=13)

# --- слой рёбер ---
folium.GeoJson(
    edges,
    name="Edges",
    style_function=lambda x: {
        "color": route_colors.get(x["properties"]["route"], "black"),
        "weight": 2
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["u", "v", "route"],
        aliases=["From:", "To:", "Route:"],
        localize=True
    )
).add_to(m)

# --- слой платформ ---
fg_platforms = folium.FeatureGroup(name="Platforms").add_to(m)
for _, row in nodes[nodes["type"] == "platform"].iterrows():
    popup_html = (
        f"<b>Node:</b> {row['node']}<br>"
        f"<b>Coords:</b> ({row.geometry.y:.5f}, {row.geometry.x:.5f})<br>"
        f"<b>Type:</b> {row['type']}<br>"
        f"<b>Route:</b> {row.get('route', '—')}"
    )
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        stroke=False, fill=True, fill_color="red", fill_opacity=1,
        popup=popup_html
    ).add_to(fg_platforms)

# --- слой остальных узлов ---
fg_other = folium.FeatureGroup(name="Other nodes").add_to(m)
for _, row in nodes[nodes["type"].notna() & (nodes["type"] != "platform")].iterrows():
    popup_html = (
        f"<b>Node:</b> {row['node']}<br>"
        f"<b>Coords:</b> ({row.geometry.y:.5f}, {row.geometry.x:.5f})<br>"
        f"<b>Type:</b> {row['type']}<br>"
        f"<b>Route:</b> {row.get('route', '—')}"
    )
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        stroke=False, fill=True, fill_color="blue", fill_opacity=1,
        popup=popup_html
    ).add_to(fg_other)

# --- слой NaN узлов ---
fg_nan = folium.FeatureGroup(name="NaN nodes").add_to(m)
for _, row in nodes[nodes["type"].isna()].iterrows():
    popup_html = (
        f"<b>Node:</b> {row['node']}<br>"
        f"<b>Coords:</b> ({row.geometry.y:.5f}, {row.geometry.x:.5f})<br>"
        f"<b>Type:</b> NaN<br>"
        f"<b>Route:</b> {row.get('route', '—')}"
    )
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        stroke=False, fill=True, fill_color="orange", fill_opacity=1,
        popup=popup_html
    ).add_to(fg_nan)

# --- контрол слоёв ---
folium.LayerControl(collapsed=False).add_to(m)

m